<a href="https://colab.research.google.com/github/kimlong77620-lgtm/web-phim-ngan/blob/main/google_colab_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set Up

1. click “Edit” -> "Notebook Settings" -> "Hardware accelerator, GPU" -> Save
<img src="https://z3.ax1x.com/2021/03/30/ciQWWV.png">

2. Click the folder icon on the left, upload your video file, and copy the uploaded video path

<img src="https://z3.ax1x.com/2021/03/30/cilvhq.png">

3. Run the code, enter the pasted video path




check whether GPU works

In [1]:
!nvidia-smi

Thu Mar 12 13:52:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc -V

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


# Install Dependencies

# Run

In [3]:
!git clone --depth=1 https://github.com/YaoFANGUK/video-subtitle-extractor.git

Cloning into 'video-subtitle-extractor'...
remote: Enumerating objects: 193, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (177/177), done.
remote: Total 193 (delta 15), reused 159 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (193/193), 964.97 MiB | 33.84 MiB/s, done.
Resolving deltas: 100% (15/15), done.
Updating files: 100% (162/162), done.


In [3]:
# Chuyển vào đúng thư mục của VSE (Bắt buộc dùng %cd)
%cd /content/video-subtitle-extractor

# Cài đặt các thư viện cần thiết
!pip install -r /content/video-subtitle-extractor/requirements.txt

# Cấu hình cài đặt
!echo -e '[DEFAULT]\nInterface = English\nLanguage = en\nMode = fast' > /content/video-subtitle-extractor/settings.ini

# Cài đặt PaddlePaddle bản mới tương thích với Colab (2.6.0)
!pip install paddlepaddle-gpu==2.6.0 -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html

/content/video-subtitle-extractor
Looking in links: https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html
  Using cached paddlepaddle_gpu-2.6.0-cp312-cp312-manylinux1_x86_64.whl.metadata (8.6 kB)
  Using cached astor-0.8.1-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached opt_einsum-3.3.0-py3-none-any.whl.metadata (6.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 749.8/749.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.6 MB/s eta 0:00:00
  Attempting uninstall: opt-einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.


In [ ]:
# Cài đặt thư viện tạo giao diện
!pip install gradio -q

import gradio as gr
import subprocess
import os
import glob
import re
import zipfile

# --- HÀM SẮP XẾP CHỈ LẤY SỐ (BỎ QUA CHỮ HÁN/VIỆT/ANH) ---
def number_only_sort_key(s):
    filename = os.path.basename(s)
    numbers = re.findall(r'\d+', filename)
    return [int(n) for n in numbers] if numbers else [0]

# --- HÀM CHÍNH: BÓC SUB HÀNG LOẠT ---
def run_vse(video_text, lang, progress=gr.Progress()):
    target_path = video_text.strip() if video_text and video_text.strip() != "" else None

    if not target_path or not os.path.exists(target_path):
        return "❌ Vui lòng dán đường dẫn File hoặc Thư mục hợp lệ từ Colab/Drive!", None

    # Thư mục gốc của phần mềm
    vse_dir = '/content/video-subtitle-extractor'

    progress(0.1, desc="Đang thiết lập hệ thống...")
    config_content = f"[DEFAULT]\nInterface = 简体中文\nLanguage = {lang}\nMode = fast\n"
    with open(os.path.join(vse_dir, "settings.ini"), "w", encoding="utf-8") as f:
        f.write(config_content)

    # 1. Trường hợp dán đường dẫn THƯ MỤC
    if os.path.isdir(target_path):
        vid_files = sorted(glob.glob(os.path.join(target_path, '*.mp4')), key=number_only_sort_key)
        if not vid_files:
            return "❌ Không tìm thấy file .mp4 nào trong thư mục này!", None

        total = len(vid_files)
        for i, vf in enumerate(vid_files):
            progress((i / total), desc=f"Đang bóc tập {i+1}/{total}: {os.path.basename(vf)}")

            # ĐÃ THÊM: env PYTHONPATH để chỉ định đúng thư mục gốc cho Python
            cmd = f"env PYTHONPATH='{vse_dir}' python backend/main.py --mp4path '{vf}' --lan '{lang}'"
            subprocess.run(cmd, shell=True, cwd=vse_dir)

        progress(0.9, desc="Đang nén toàn bộ phụ đề thành file ZIP...")
        zip_name = "Toan_Bo_Phu_De.zip"

        srt_files = glob.glob(os.path.join(target_path, '*.srt'))

        with zipfile.ZipFile(zip_name, 'w') as zipf:
            for srt in srt_files:
                zipf.write(srt, os.path.basename(srt))

        return f"✅ Đã quét xong {total} video! Hãy tải file ZIP chứa toàn bộ phụ đề ở dưới.", zip_name

    # 2. Trường hợp dán đường dẫn 1 FILE LẺ
    else:
        progress(0.4, desc="AI đang bóc tách phụ đề (Vui lòng đợi)...")

        # ĐÃ THÊM: env PYTHONPATH để chỉ định đúng thư mục gốc cho Python
        cmd = f"env PYTHONPATH='{vse_dir}' python backend/main.py --mp4path '{target_path}' --lan '{lang}'"

        process = subprocess.run(cmd, shell=True, cwd=vse_dir, capture_output=True, text=True)

        target_dir = os.path.dirname(target_path)
        srt_files = glob.glob(os.path.join(target_dir, '*.srt'))

        if not srt_files:
            error_log = process.stderr if process.stderr else process.stdout
            return f"⚠️ Không thấy file SRT. Đây là báo cáo của AI:\n{error_log}", None

        latest_srt = max(srt_files, key=os.path.getctime)
        return "✅ Bóc sub thành công!", latest_srt

# --- XÂY DỰNG GIAO DIỆN WEB ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 🎬 TRẠM 1: BÓC PHỤ ĐỀ HÀNG LOẠT (Tự động & Xếp chuẩn số)")

    gr.Markdown("**Mẹo:** Dán đường dẫn **FILE** hoặc **THƯ MỤC** từ Google Drive vào ô dưới. Nếu dán thư mục, AI tự động gom toàn bộ file SRT vào 1 file `.zip`.")

    with gr.Row():
        with gr.Column(scale=3):
            vid_path_1 = gr.Textbox(label="Dán đường dẫn FILE hoặc THƯ MỤC (Lấy từ thanh bên trái Colab)", placeholder="VD: /content/drive/MyDrive/Phim_Bo_A/")
            lang_input = gr.Dropdown(choices=["ch", "en", "ko", "ja"], value="ch", label="Ngôn ngữ gốc")

        with gr.Column(scale=2):
            txt_output_1 = gr.Textbox(label="Trạng thái (Log)", lines=6)
            with gr.Row():
                btn_extract = gr.Button("🚀 Chạy Bóc Sub", variant="primary")
                file_output_1 = gr.File(label="Tải file SRT / ZIP về máy")

    btn_extract.click(fn=run_vse, inputs=[vid_path_1, lang_input], outputs=[txt_output_1, file_output_1])

# Khởi chạy giao diện
demo.launch(share=True, debug=True)

Here is an example:

input video path: /content/video-subtitle-extractor/test/test_en.mp4

input subtitle area: 612 717 90 1191

In [ ]:
!python ./backend/main.py